In [ ]:
# ==========================================
# FAKE GOOGLE COLAB & OPENAI MOCK FOR LOCAL RUN
# ==========================================
import sys
from types import ModuleType
import os

# Set API keys
os.environ["GROQ_API_KEY"] = os.environ.get("GROQ_API_KEY", "")
os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN", "")
os.environ["OPENROUTER_API_KEY"] = os.environ.get("GROQ_API_KEY", "")
os.environ["OPENAI_API_KEY"] = os.environ.get("GROQ_API_KEY", "")

class FakeUserdata:
    def get(self, key):
        if key == "GROQ_API_KEY":
            return os.environ.get("GROQ_API_KEY", "")
        if key == "HF_TOKEN":
            return os.environ.get("HF_TOKEN", "")
        if key in ["OPENROUTER_API_KEY", "OPENAI_API_KEY"]:
            return os.environ.get("GROQ_API_KEY", "")
        return os.environ.get(key)

class FakeFiles:
    def upload(self):
        print("MOCK: files.upload() - using local 'anual report mini proj 01.pdf'")
        return {"anual report mini proj 01.pdf": b""}

# Inject google.colab
colab_module = ModuleType("google.colab")
colab_module.userdata = FakeUserdata()
colab_module.files = FakeFiles()
sys.modules["google.colab"] = colab_module

# Patch openai to redirect to Groq (with recursion guard)
import openai
from openai import OpenAI

if not getattr(OpenAI, "_is_patched", False):
    orig_init = OpenAI.__init__
    def new_init(self, *args, **kwargs):
        kwargs["api_key"] = os.environ.get("GROQ_API_KEY", "")
        kwargs["base_url"] = "https://api.groq.com/openai/v1"
        orig_init(self, *args, **kwargs)
    OpenAI.__init__ = new_init
    OpenAI._is_patched = True

# Map models in Chat Completions (with recursion guard)
from openai.resources.chat.completions import Completions

if not getattr(Completions, "_is_patched", False):
    orig_create = Completions.create
    def new_create(self, *args, **kwargs):
        model = kwargs.get("model")
        if model in ["gpt-4o", "gpt-4o-mini", "gpt-4-turbo", "openai/gpt-4o", "openai/gpt-4o-mini"]:
            kwargs["model"] = "llama-3.3-70b-versatile"
        elif model in ["llama-3.1-8b-instant", "meta-llama/llama-3.1-8b-instant"]:
            kwargs["model"] = "llama-3.1-8b-instant"
        
        # If the response format is JSON, force compatible settings
        if "response_format" in kwargs and isinstance(kwargs["response_format"], dict):
            if kwargs["response_format"].get("type") == "json_object":
                pass # Keep it
                
        return orig_create(self, *args, **kwargs)
    Completions.create = new_create
    Completions._is_patched = True

print("MOCK: google.colab and openai patched successfully!")


# 📚 Mini Project 01 — Part 3: "The Librarian" (Advanced RAG System)
### Operation Ledger-Mind | AI Engineer Essentials

**Goal:** Build a search-based system that finds the exact right passage in Uber's 2024
Annual Report and uses it to answer questions accurately, with no training required.

**What you'll build:**
- A Weaviate vector database storing all your PDF chunks
- Hybrid search: Dense vector search (semantic meaning) + BM25 (exact keyword matching)
- Reciprocal Rank Fusion (RRF) to combine both search results into one ranked list
- Cross-Encoder reranking to polish the final ranking
- A function `query_librarian(question)` that ties it all together

### Before you start
1. No GPU needed for this notebook — a standard CPU runtime is fine
2. You'll need an OpenAI API key (same one from Part 1) to generate the final answer
3. We reuse the **same chunks** from Part 1, so make sure that notebook has been run first

---


## Step 0 — Understanding the Pipeline (read this first!)

RAG stands for "Retrieval-Augmented Generation" — instead of the model memorizing facts,
we retrieve the relevant text first, then hand it to the model to read and answer from.

Here's the full pipeline we're building, in order:

| Stage | What it does | Why we need it |
|-------|--------------|-----------------|
| **1. Chunking** | Split the PDF into small pieces | Already done in Part 1 — we reuse those chunks |
| **2. Embedding** | Convert each chunk into a list of numbers (a "vector") that represents its meaning | Lets us search by *meaning*, not just exact words |
| **3. Weaviate storage** | Store chunks + vectors in a searchable database | Fast retrieval even across thousands of chunks |
| **4. Dense search** | Find chunks whose *meaning* is similar to the question | Good at understanding paraphrased questions |
| **5. BM25 search** | Find chunks containing the *exact keywords* from the question | Good at catching specific terms like "Form 10-K" or "$37B" that semantic search can miss |
| **6. RRF fusion** | Merge the two ranked lists into one combined ranking | Gets the best of both search types |
| **7. Cross-Encoder rerank** | Re-score the top candidates more carefully | Cleans up the final order for higher precision |
| **8. Answer generation** | Feed the best chunks + question to an LLM | Produces the final, grounded answer |

**Why hybrid search instead of just one method?** Dense vector search is great at understanding
that "How much money did Uber make?" means the same thing as "What was Uber's revenue?" — but
it can blur over exact numbers and codes. BM25 is the opposite: it's excellent at finding exact
strings like "10-K" or "$43.98 billion" but doesn't understand paraphrasing at all. Combining
both covers each other's blind spots — which is exactly why financial documents need this.


## Step 1 — Install Required Libraries

We're installing:
- `weaviate-client` — talks to the Weaviate vector database
- `sentence-transformers` — generates embeddings and runs the cross-encoder reranker
- `openai` — generates the final answer
- `rank-bm25` — a lightweight Python BM25 implementation we use alongside Weaviate's own


In [2]:
!pip install -q weaviate-client sentence-transformers openai rank-bm25 tqdm

print("Libraries installed!")


zsh:1: command not found: pip


Libraries installed!


## Step 2 — Imports, API Keys & Configuration

Paste your OpenAI key (same one from Part 1) below.


In [3]:
import json
import re
import time
from pathlib import Path

import weaviate
from weaviate.classes.config import Configure, Property, DataType
from weaviate.classes.query import MetadataQuery
from sentence_transformers import CrossEncoder, SentenceTransformer
from rank_bm25 import BM25Okapi
from openai import OpenAI
from tqdm import tqdm
from google.colab import userdata

# --------------------------------------------------------------------
# Configure API keys and models
# --------------------------------------------------------------------
# OpenRouter API key for LLM calls (Gemini, GPT-4, etc.)
OPENROUTER_API_KEY = userdata.get('OPENROUTER_API_KEY')

# Local embedding model for Weaviate (no OpenAI key needed for embeddings)
sentence_transformer_model = SentenceTransformer("BAAI/bge-small-en-v1.5")

# Model used to generate the final answer via OpenRouter
ANSWER_MODEL = "gpt-4o-mini"

# Path to where Part 1 saved its chunks (we reconstruct chunks from train+test jsonl,
# OR you can re-run the PDF chunking code directly -- both options are shown in Step 3)
TRAIN_JSONL_PATH = "train.jsonl"
TEST_JSONL_PATH  = "golden_test_set.jsonl"

# How many results to retrieve from each search method before fusion
TOP_K_DENSE = 10
TOP_K_BM25  = 10

# How many results to keep after RRF fusion, before reranking
TOP_K_FUSION = 10

# How many final chunks to feed to the LLM after reranking
TOP_K_FINAL = 3

# Configure OpenAI client to use OpenRouter
client_openai = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
)

print("Configuration set!")
print(f"   Answer model      : {ANSWER_MODEL}")
print(f"   Dense top-k        : {TOP_K_DENSE}")
print(f"   BM25 top-k         : {TOP_K_BM25}")
print(f"   Final chunks to LLM: {TOP_K_FINAL}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/94.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Configuration set!
   Answer model      : gpt-4o-mini
   Dense top-k        : 10
   BM25 top-k         : 10
   Final chunks to LLM: 3


## Step 3 — Load the Chunks from Part 1

The Librarian needs the same source chunks that The Intern was trained on, so both systems
are competing fairly on identical source material.

We pull the unique chunks back out of your `train.jsonl` and `golden_test_set.jsonl` files
from Part 1 (each chunk's text was already stored there inside the user message). If you
don't have those files, the fallback cell re-extracts chunks directly from the PDF.


In [4]:
from google.colab import files

uploaded = files.upload()

MOCK: files.upload() - using local 'anual report mini proj 01.pdf'


In [5]:
import os

print(os.path.exists("anual report mini proj 01.pdf"))

True


In [6]:
!pip install -q pdfplumber

import pdfplumber
import re

def clean_page_text(text):
    if not text:
        return ""

    text = re.sub(r"[\u2612\u2610]", "", text)
    text = re.sub(r"^\s*\d{1,3}\s*$", "", text, flags=re.MULTILINE)
    text = re.sub(r"^\s*[a-zA-Z]\s*$", "", text, flags=re.MULTILINE)
    text = re.sub(r"Table of Contents", "", text, flags=re.IGNORECASE)
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()


def chunk_text(text, chunk_size=1500, overlap=200):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]

        if len(chunk.strip()) > 100:
            chunks.append(chunk.strip())

        start += (chunk_size - overlap)

    return chunks


PDF_PATH = "anual report mini proj 01.pdf"

with pdfplumber.open(PDF_PATH) as pdf:
    pages = [clean_page_text(page.extract_text()) for page in pdf.pages]

full_text = "\n\n".join(page for page in pages if page)

raw_chunks = chunk_text(full_text)

chunks_data = [
    {"chunk_id": i, "text": chunk}
    for i, chunk in enumerate(raw_chunks)
]

print(f"Rebuilt {len(chunks_data)} chunks directly from PDF")

/opt/homebrew/Cellar/python@3.14/3.14.6/Frameworks/Python.framework/Versions/3.14/lib/python3.14/pty.py:66: DeprecationWarning: This process (pid=83402) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


zsh:1: command not found: pip


Rebuilt 477 chunks directly from PDF


In [7]:
import pdfplumber
import re

PDF_PATH = "anual report mini proj 01.pdf"

with pdfplumber.open(PDF_PATH) as pdf:
    pages = []

    for page in pdf.pages:
        text = page.extract_text()
        if text:
            pages.append(text)

print("Pages extracted:", len(pages))

full_text = "\n\n".join(pages)

print("Characters:", len(full_text))


def chunk_text(text, chunk_size=1500, overlap=200):
    chunks = []
    start = 0

    while start < len(text):
        chunk = text[start:start+chunk_size]

        if chunk.strip():
            chunks.append(chunk)

        start += chunk_size - overlap

    return chunks


raw_chunks = chunk_text(full_text)

chunks_data = [
    {"chunk_id": i, "text": c}
    for i, c in enumerate(raw_chunks)
]

print("Chunks:", len(chunks_data))

Pages extracted: 142
Characters: 620407
Chunks: 478


**Fallback option:** if `train.jsonl` / `golden_test_set.jsonl` aren't available, run the
cell below instead to rebuild chunks directly from the PDF (same logic as Part 1, Step 2-3).


## Step 4 — Start Weaviate & Create the Schema

**What is Weaviate?** It's a vector database — a system specifically built to store text
embeddings and search through them efficiently, even across millions of items.

We use **Weaviate Embedded**, which runs Weaviate directly inside your Colab notebook
(no separate server or sign-up needed). If Embedded mode causes issues in your environment,
the assignment allows using Weaviate Cloud (WCD) free tier instead — see the note below.

**The schema** defines what fields each chunk will have when stored: the text itself, the
chunk ID, and Weaviate will automatically generate the vector embedding for us.


In [8]:
import os

# Set a dummy OpenAI API key as an environment variable to satisfy Weaviate's internal check,
# even though we are using local embeddings and Configure.Vectorizer.none().
# This is a workaround for an unexpected behavior in Weaviate Embedded.
os.environ["OPENAI_API_KEY"] = "sk-dummy-key-for-weaviate"

# --------------------------------------------------------------------
# Connect to Weaviate (Embedded mode -- runs locally inside Colab)
# --------------------------------------------------------------------
# The previous attempt to connect to embedded failed because ports were in use.
# Connecting to the local instance directly.
weaviate_client = weaviate.connect_to_embedded(port=8079, grpc_port=50050)

print("Connected to Weaviate (Local mode)")
print(f"   Weaviate is ready: {weaviate_client.is_ready()}")

# --------------------------------------------------------------------
# NOTE: If Embedded mode fails in your environment, use Weaviate Cloud instead:
#
# weaviate_client = weaviate.connect_to_weaviate_cloud(
#     cluster_url="YOUR_WCD_CLUSTER_URL",
#     auth_credentials=weaviate.auth.AuthApiKey("YOUR_WCD_API_KEY"),
#     headers={
#         "X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"] # Use the dummy key for Weaviate Cloud if needed
#     },
# )
# --------------------------------------------------------------------

{"build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","level":"warning","log_level_env":"","msg":"log level not recognized, defaulting to info","time":"2026-07-15T01:19:48+05:30"}
{"action":"startup","build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","level":"info","msg":"Feature flag LD integration disabled: could not locate WEAVIATE_LD_API_KEY env variable","time":"2026-07-15T01:19:48+05:30"}
{"action":"startup","build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","default_vectorizer_module":"none","level":"info","msg":"the default vectorizer modules is set to \"none\", as a result all new schema classes without an explicit vectorizer setting, will use this vectorizer","time":"2026-07-15T01:19:48+05:30"}
{"action":"startup","auto_schema_enabled":{},"build_git_commit":"62dcafac32","build_go_version":

{"action":"read_disk_use","build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","level":"warning","msg":"disk usage currently at 82.65%, threshold set to 80.00%","path":"/Users/kavishkathilakarathna/.local/share/weaviate","time":"2026-07-15T01:19:49+05:30"}


{"build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","level":"info","msg":"attempting to join","remoteNodes":{"Embedded_at_8079":"192.168.1.17:56510"},"time":"2026-07-15T01:19:49+05:30"}
{"build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","level":"info","msg":"attempted to join and failed","remoteNode":"192.168.1.17:56510","status":8,"time":"2026-07-15T01:19:49+05:30"}


{"action":"raft","build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","last-leader-addr":"","last-leader-id":"","level":"warning","msg":"heartbeat timeout reached, starting election","time":"2026-07-15T01:19:50+05:30"}
{"action":"raft","build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","level":"info","msg":"entering candidate state","node":{},"term":3,"time":"2026-07-15T01:19:50+05:30"}
{"action":"raft","build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","level":"info","msg":"pre-vote successful, starting election","refused":0,"tally":1,"term":3,"time":"2026-07-15T01:19:50+05:30","votesNeeded":1}
{"action":"raft","build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","level":"info","msg":"election won","tally":1,"term":3,"time":"2026-07-15T0

{"build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","level":"info","msg":"configured versions","server_version":"1.30.5","time":"2026-07-15T01:19:50+05:30","version":"1.30.5"}
{"action":"grpc_startup","build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","level":"info","msg":"grpc server listening at [::]:50050","time":"2026-07-15T01:19:50+05:30"}
{"address":"192.168.1.17:56510","build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","level":"info","msg":"current Leader","time":"2026-07-15T01:19:50+05:30"}
{"build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","level":"info","msg":"starting migration from old schema","time":"2026-07-15T01:19:50+05:30"}
{"build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build

{"action":"telemetry_push","build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","level":"info","msg":"telemetry started","payload":"\u0026{MachineID:fa2af8b6-2dbf-4dfd-9d20-00ce5296efc7 Type:INIT Version:1.30.5 ObjectsCount:0 OS:darwin Arch:arm64 UsedModules:[] CollectionsCount:0}","time":"2026-07-15T01:19:53+05:30"}


Connected to Weaviate (Local mode)
   Weaviate is ready: True


In [9]:
COLLECTION_NAME = "UberReportChunks"

# Delete the collection if it already exists (clean slate for re-runs)
if weaviate_client.collections.exists(COLLECTION_NAME):
    weaviate_client.collections.delete(COLLECTION_NAME)
    print(f"Deleted existing collection: {COLLECTION_NAME}")

# --------------------------------------------------------------------
# Create the schema
# vectorizer_config tells Weaviate to use no vectorizer, we'll provide vectors directly
# --------------------------------------------------------------------
collection = weaviate_client.collections.create(
    name=COLLECTION_NAME,
    vectorizer_config=Configure.Vectorizer.none(), # We provide our own vectors
    vector_index_config=Configure.VectorIndex.flat(), # Use vector_index_config for the default vector
    properties=[
        Property(name="chunk_id", data_type=DataType.INT),
        Property(name="text", data_type=DataType.TEXT),
    ],
)

print(f"Schema created: collection '{COLLECTION_NAME}' is ready for data")

Schema created: collection 'UberReportChunks' is ready for data


/Users/kavishkathilakarathna/Desktop/AEE Pro01/.venv/lib/python3.14/site-packages/weaviate/warnings.py:196: DeprecationWarning: Dep024: You are using the `vectorizer_config` argument in `collection.config.create()`, which is deprecated.
            Use the `vector_config` argument instead.
            
  warnings.warn(
/Users/kavishkathilakarathna/Desktop/AEE Pro01/.venv/lib/python3.14/site-packages/weaviate/warnings.py:206: DeprecationWarning: Dep025: You are using the `vector_index_config` argument in `collection.config.create()`, which is deprecated.
            Use the `vector_config` argument instead defining `vector_index_config` as a sub-argument.
            
  warnings.warn(


## Step 5 — Index All Chunks into Weaviate

This step uploads every chunk into Weaviate. As each chunk goes in, Weaviate automatically
calls OpenAI's embedding API in the background to generate its dense vector representation.

We use **batch insertion** to do this efficiently rather than one-by-one.


In [10]:
collection = weaviate_client.collections.get(COLLECTION_NAME)

print(f"Indexing {len(chunks_data)} chunks into Weaviate...")

# Generate embeddings for all chunks locally
print("Generating embeddings locally...")
chunk_texts = [chunk["text"] for chunk in chunks_data]
embeddings = sentence_transformer_model.encode(chunk_texts, normalize_embeddings=True)
print("Embeddings generated!")

with collection.batch.dynamic() as batch:
    for i, chunk in tqdm(enumerate(chunks_data), desc="Indexing", total=len(chunks_data)):
        batch.add_object(
            properties={
                "chunk_id": chunk["chunk_id"],
                "text": chunk["text"],
            },
            vector=embeddings[i] # Provide the pre-calculated vector
        )

# Check for any failed insertions
failed = collection.batch.failed_objects
if failed:
    print(f"WARNING: {len(failed)} objects failed to index")
else:
    print("All chunks indexed successfully!")

# Confirm count
total_in_db = collection.aggregate.over_all(total_count=True).total_count
print(f"   Total objects in Weaviate: {total_in_db}")

Indexing 478 chunks into Weaviate...
Generating embeddings locally...


{"build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","level":"warning","msg":"prop len tracker file /Users/kavishkathilakarathna/.local/share/weaviate/uberreportchunks/LNr26dx0zk89/proplengths does not exist, creating new tracker","time":"2026-07-15T01:20:00+05:30"}


{"build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","level":"info","msg":"Created shard uberreportchunks_LNr26dx0zk89 in 257.322042ms","time":"2026-07-15T01:20:00+05:30"}


{"action":"read_disk_use","build_git_commit":"62dcafac32","build_go_version":"go1.24.3","build_image_tag":"HEAD","build_wv_version":"1.30.5","level":"warning","msg":"disk usage currently at 82.66%, threshold set to 80.00%","path":"/Users/kavishkathilakarathna/.local/share/weaviate","time":"2026-07-15T01:20:19+05:30"}


Embeddings generated!


Indexing:   0%|          | 0/478 [00:00<?, ?it/s]

Indexing:  50%|█████     | 240/478 [00:00<00:00, 1449.35it/s]

Indexing: 100%|██████████| 478/478 [00:00<00:00, 2059.77it/s]

All chunks indexed successfully!
   Total objects in Weaviate: 478


## Step 6 — Build the BM25 Keyword Search Index

BM25 is a classic keyword-matching algorithm (it's what older search engines like the
original Elasticsearch used by default). Unlike dense vectors, it doesn't understand meaning —
it just scores documents based on how often and how uniquely your query words appear in them.

We build this as a **separate, local index** using the `rank_bm25` library so we have full
control over combining it with Weaviate's dense search results in Step 8 (RRF fusion).

*Note: Weaviate also has a built-in BM25/hybrid search feature. We're using a standalone
BM25 index here so the RRF fusion logic is fully transparent and easy to explain in your
engineering report.*


In [11]:
def simple_tokenize(text):
    # Lowercase and split on non-alphanumeric characters
    return re.findall(r"\b\w+\b", text.lower())


# Tokenize every chunk for BM25
tokenized_chunks = [simple_tokenize(chunk["text"]) for chunk in chunks_data]

# Build the BM25 index
bm25_index = BM25Okapi(tokenized_chunks)

print(f"BM25 index built over {len(chunks_data)} chunks")


BM25 index built over 478 chunks


## Step 7 — Dense Search & BM25 Search Functions

Now we write two separate search functions:

- `dense_search(query)` — asks Weaviate to find chunks whose *meaning* is closest to the query
- `bm25_search(query)` — asks our local BM25 index to find chunks with the most *keyword overlap*

Both return a ranked list of `(chunk_id, score)` so we can combine them in the next step.


In [12]:
def dense_search(query, top_k=TOP_K_DENSE):
    # Semantic search using Weaviate vector similarity with locally generated embeddings.
    # Returns a list of chunk_ids ranked from most to least relevant.

    # Generate embedding for the query using the local model
    query_embedding = sentence_transformer_model.encode(query, normalize_embeddings=True).tolist()

    response = collection.query.near_vector(
        near_vector=query_embedding,
        limit=top_k,
        return_metadata=MetadataQuery(distance=True),
    )

    results = []
    for obj in response.objects:
        results.append({
            "chunk_id": obj.properties["chunk_id"],
            "text": obj.properties["text"],
            "distance": obj.metadata.distance,   # lower distance = more similar
        })
    return results


def bm25_search(query, top_k=TOP_K_BM25):
    # Keyword search using BM25.
    # Returns a list of chunk_ids ranked from most to least relevant.
    tokenized_query = simple_tokenize(query)
    scores = bm25_index.get_scores(tokenized_query)

    # Pair each chunk with its score, sort descending
    scored_chunks = list(zip(chunks_data, scores))
    scored_chunks.sort(key=lambda x: x[1], reverse=True)

    results = []
    for chunk, score in scored_chunks[:top_k]:
        results.append({
            "chunk_id": chunk["chunk_id"],
            "text": chunk["text"],
            "score": score,
        })
    return results


print("Search functions ready: dense_search() and bm25_search()")

Search functions ready: dense_search() and bm25_search()


Let's quickly test both search methods on the same query to see how they differ.


In [13]:
from google.colab import userdata
userdata.get('OPENROUTER_API_KEY')

'YOUR_GROQ_API_KEY'

In [14]:
test_query = "What was Uber total revenue in 2024?"

print(f"QUERY: {test_query}\n")

print("-- Dense search results (top 3) --")
for r in dense_search(test_query, top_k=3):
    print(f"  chunk {r['chunk_id']} (distance={r['distance']:.4f}): {r['text'][:100]}...")

print("\n-- BM25 search results (top 3) --")
for r in bm25_search(test_query, top_k=3):
    print(f"  chunk {r['chunk_id']} (score={r['score']:.4f}): {r['text'][:100]}...")


QUERY: What was Uber total revenue in 2024?

-- Dense search results (top 3) --


  chunk 241 (distance=0.1891): s of our consolidated statements of operations for each of the periods presented as a
percentage of ...
  chunk 362 (distance=0.2253): ective for public companies for fiscal years beginning after December
93

15, 2026, and interim peri...
  chunk 226 (distance=0.2558): entary Data,” of
this Annual Report on Form 10-K.
In addition, if we are required to classify Driver...

-- BM25 search results (top 3) --
  chunk 241 (score=11.4489): s of our consolidated statements of operations for each of the periods presented as a
percentage of ...
  chunk 0 (score=10.5270): On Our Way
2024 ANNUAL REPORT

Uber’s Mission
We reimagine the way the world moves for the better
We...
  chunk 302 (score=9.9713): g-term insurance reserves 4,909 7,042
Long-term debt, net of current portion 9,459 8,347
Operating l...


## Step 8 — Reciprocal Rank Fusion (RRF)

**The problem RRF solves:** Dense search gives you a "distance" score, BM25 gives you a
totally different kind of score — these numbers aren't on the same scale, so you can't just
average them directly.

**How RRF works:** Instead of using the raw scores, it only looks at **rank position**
(1st place, 2nd place, etc.) in each list. Each chunk gets a fusion score:

```
RRF_score(chunk) = sum over each search method of:  1 / (k + rank_in_that_method)
```

`k` is a constant (commonly 60) that smooths out the effect of rank differences. A chunk that
appears near the top of *both* lists gets a high combined score. A chunk that's only in one
list still gets credit, just less.


In [15]:
def reciprocal_rank_fusion(dense_results, bm25_results, k=60):
    # Combine dense and BM25 ranked lists into one fused ranking using RRF.
    #
    # Args:
    #     dense_results : list of dicts from dense_search(), ordered best-first
    #     bm25_results  : list of dicts from bm25_search(), ordered best-first
    #     k             : RRF smoothing constant (60 is the standard default)
    #
    # Returns:
    #     List of dicts sorted by fused RRF score, best first.

    rrf_scores = {}      # chunk_id -> accumulated RRF score
    chunk_texts = {}     # chunk_id -> text (so we can look it up later)

    # Add scores from dense search ranking
    for rank, result in enumerate(dense_results, start=1):
        cid = result["chunk_id"]
        rrf_scores[cid] = rrf_scores.get(cid, 0) + 1 / (k + rank)
        chunk_texts[cid] = result["text"]

    # Add scores from BM25 search ranking
    for rank, result in enumerate(bm25_results, start=1):
        cid = result["chunk_id"]
        rrf_scores[cid] = rrf_scores.get(cid, 0) + 1 / (k + rank)
        chunk_texts[cid] = result["text"]

    # Sort all chunks by their combined RRF score, descending
    fused = [
        {"chunk_id": cid, "text": chunk_texts[cid], "rrf_score": score}
        for cid, score in rrf_scores.items()
    ]
    fused.sort(key=lambda x: x["rrf_score"], reverse=True)

    return fused


# -- Test it on the same query --
dense_r = dense_search(test_query, top_k=TOP_K_DENSE)
bm25_r  = bm25_search(test_query, top_k=TOP_K_BM25)
fused_r = reciprocal_rank_fusion(dense_r, bm25_r)

print(f"QUERY: {test_query}\n")
print(f"-- Fused results (top {TOP_K_FUSION}) --")
for r in fused_r[:TOP_K_FUSION]:
    print(f"  chunk {r['chunk_id']} (rrf_score={r['rrf_score']:.4f}): {r['text'][:100]}...")


QUERY: What was Uber total revenue in 2024?

-- Fused results (top 10) --
  chunk 241 (rrf_score=0.0328): s of our consolidated statements of operations for each of the periods presented as a
percentage of ...
  chunk 362 (rrf_score=0.0313): ective for public companies for fiscal years beginning after December
93

15, 2026, and interim peri...
  chunk 302 (rrf_score=0.0310): g-term insurance reserves 4,909 7,042
Long-term debt, net of current portion 9,459 8,347
Operating l...
  chunk 0 (rrf_score=0.0161): On Our Way
2024 ANNUAL REPORT

Uber’s Mission
We reimagine the way the world moves for the better
We...
  chunk 226 (rrf_score=0.0159): entary Data,” of
this Annual Report on Form 10-K.
In addition, if we are required to classify Driver...
  chunk 254 (rrf_score=0.0156): g any applicable taxes, tolls, and fees, of: Mobility
rides, Delivery orders (in each case without a...
  chunk 414 (rrf_score=0.0156): ith market-based targets granted in the years ended December 31, 2022, 2023 and 

## Step 9 — Cross-Encoder Reranking

**Why rerank after RRF?** Dense search and BM25 both compare the query against each chunk
*independently* and somewhat shallowly. A **cross-encoder** is a more powerful (but slower)
model that looks at the query and a candidate chunk **together, side by side**, and produces
a single relevance score. It's much more accurate, but too slow to run on thousands of chunks —
so we only run it on the small shortlist that survived RRF fusion (10 candidates).

This is a common two-stage pattern in production search systems: cast a wide net cheaply
(dense + BM25), then carefully re-score just the finalists (cross-encoder).

We use `cross-encoder/ms-marco-MiniLM-L-6-v2`, a small, fast, well-tested reranking model.


In [16]:
print("Loading cross-encoder model (first run downloads ~80MB)...")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Cross-encoder loaded!")


Loading cross-encoder model (first run downloads ~80MB)...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-encoder loaded!


In [17]:
def cross_encoder_rerank(query, candidates, top_k=TOP_K_FINAL):
    # Rerank a shortlist of candidate chunks using a cross-encoder model.
    #
    # Args:
    #     query      : the question being asked
    #     candidates : list of dicts (from RRF fusion), each with a "text" field
    #     top_k      : how many top results to return after reranking
    #
    # Returns:
    #     List of dicts sorted by cross-encoder relevance score, best first.

    # Cross-encoders take (query, document) PAIRS as input
    pairs = [(query, c["text"]) for c in candidates]

    # Score every pair in one batch call
    scores = cross_encoder.predict(pairs)

    # Attach scores back to their candidates
    for candidate, score in zip(candidates, scores):
        candidate["rerank_score"] = float(score)

    # Sort by the new cross-encoder score, descending
    reranked = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)

    return reranked[:top_k]


# -- Test it --
final_results = cross_encoder_rerank(test_query, fused_r[:TOP_K_FUSION], top_k=TOP_K_FINAL)

print(f"QUERY: {test_query}\n")
print(f"-- Final reranked results (top {TOP_K_FINAL}) --")
for r in final_results:
    print(f"  chunk {r['chunk_id']} (rerank_score={r['rerank_score']:.4f}): {r['text'][:120]}...")


QUERY: What was Uber total revenue in 2024?

-- Final reranked results (top 3) --
  chunk 362 (rerank_score=1.9549): ective for public companies for fiscal years beginning after December
93

15, 2026, and interim periods within fiscal ye...
  chunk 412 (rerank_score=1.0957): ith acquisitions. Vesting
of this stock may be dependent on a combination of service and performance conditions that bec...
  chunk 241 (rerank_score=0.9519): s of our consolidated statements of operations for each of the periods presented as a
percentage of revenue (1):
Year En...


## Step 10 — Generate the Final Answer

Now that we have the best 3 chunks, we feed them to an LLM along with the original question.
This is the "Generation" part of "Retrieval-Augmented Generation" — the model only has to
read and synthesize, not recall from memory, which dramatically reduces hallucination.

We also instruct the model to **cite which chunk** it used, so you get traceable, page-level
citations — exactly the value proposition "The Librarian" promises in the assignment brief.


In [18]:
ANSWER_GENERATION_PROMPT = """You are a senior financial analyst at a quantitative hedge fund.
Answer the question using ONLY the information in the passages below.

RULES:
- Base your answer strictly on the provided passages -- do not use outside knowledge
- Be precise with numbers, dates, and names -- quote them exactly as they appear
- If the passages do not contain enough information, say so clearly
- After your answer, list which passage number(s) you used as [Source: Passage X]

PASSAGES:
{passages}

QUESTION: {question}

ANSWER:"""


def generate_answer_from_context(question, retrieved_chunks):
    # Format the retrieved chunks into a numbered passages block
    passages_text = "\n\n".join([
        f"--- Passage {i+1} (chunk_id={c['chunk_id']}) ---\n{c['text']}"
        for i, c in enumerate(retrieved_chunks)
    ])

    prompt = ANSWER_GENERATION_PROMPT.format(passages=passages_text, question=question)

    response = client_openai.chat.completions.create(
        model=ANSWER_MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2,   # low temperature -- we want grounded, factual answers
        max_tokens=500,
    )

    return response.choices[0].message.content.strip()


print("Answer generation function ready!")


Answer generation function ready!


## Step 11 — Build `query_librarian(question)`

This is the required deliverable — one function that runs the **entire pipeline** end to end:

dense search + BM25 search → RRF fusion → cross-encoder rerank → LLM answer generation


In [19]:
def query_librarian(question, verbose=False):
    # Ask "The Librarian" a question. Runs the full hybrid RAG pipeline:
    #   1. Dense vector search (Weaviate)
    #   2. BM25 keyword search
    #   3. Reciprocal Rank Fusion (combine the two rankings)
    #   4. Cross-Encoder reranking (polish the final ranking)
    #   5. LLM answer generation grounded in the top chunks
    #
    # Args:
    #     question : the question to ask
    #     verbose  : if True, prints each pipeline stage intermediate results
    #
    # Returns:
    #     dict with "answer", "sources" (chunk_ids used), and "latency_ms"

    start_time = time.time()

    # Stage 1 + 2: retrieve candidates from both search methods
    dense_results = dense_search(question, top_k=TOP_K_DENSE)
    bm25_results  = bm25_search(question, top_k=TOP_K_BM25)

    if verbose:
        print(f"Dense search found {len(dense_results)} candidates")
        print(f"BM25 search found {len(bm25_results)} candidates")

    # Stage 3: fuse rankings with RRF
    fused_results = reciprocal_rank_fusion(dense_results, bm25_results)
    fused_shortlist = fused_results[:TOP_K_FUSION]

    if verbose:
        print(f"RRF fusion produced {len(fused_shortlist)} shortlisted candidates")

    # Stage 4: cross-encoder rerank the shortlist
    final_chunks = cross_encoder_rerank(question, fused_shortlist, top_k=TOP_K_FINAL)

    if verbose:
        print(f"Cross-encoder selected final top {len(final_chunks)} chunks:")
        for c in final_chunks:
            print(f"   chunk {c['chunk_id']} (score={c['rerank_score']:.4f})")

    # Stage 5: generate the final answer
    answer = generate_answer_from_context(question, final_chunks)

    latency_ms = (time.time() - start_time) * 1000

    return {
        "answer": answer,
        "sources": [c["chunk_id"] for c in final_chunks],
        "latency_ms": latency_ms,
    }


print("query_librarian() is ready to use!")


query_librarian() is ready to use!


## Step 12 — Test The Librarian

Let's test it with the same questions used to test The Intern in Part 2, so you can directly
compare the two systems' answers later in Part 4 (The Showdown).


In [20]:
test_questions = [
    "What was Uber total revenue for fiscal year 2024?",
    "What are Uber main strategic priorities mentioned in the report?",
    "Write a 2-sentence summary of Uber business for a non-technical investor.",
]

for q in test_questions:
    print(f"QUESTION: {q}")
    result = query_librarian(q, verbose=False)
    print(f"LIBRARIAN ANSWER: {result['answer']}")
    print(f"Sources used: chunk_ids {result['sources']}")
    print(f"Latency: {result['latency_ms']:.0f} ms")
    print("\n" + "-" * 80 + "\n")


QUESTION: What was Uber total revenue for fiscal year 2024?


LIBRARIAN ANSWER: Uber's total revenue for fiscal year 2024 was $43,978 million.

[Source: Passage 1, Passage 2]
Sources used: chunk_ids [240, 362, 241]
Latency: 2853 ms

--------------------------------------------------------------------------------

QUESTION: What are Uber main strategic priorities mentioned in the report?


LIBRARIAN ANSWER: The passages provided do not explicitly mention Uber's main strategic priorities. They discuss various aspects of Uber's business, such as its cybersecurity program, legal proceedings, and company information, but do not outline specific strategic priorities.

[Source: Passage 1, Passage 2, Passage 3]
Sources used: chunk_ids [212, 223, 32]
Latency: 1452 ms

--------------------------------------------------------------------------------

QUESTION: Write a 2-sentence summary of Uber business for a non-technical investor.


LIBRARIAN ANSWER: Uber Technologies, Inc. is a technology platform that uses a network, technology, and expertise to power movement from one place to another. The company, headquartered in San Francisco, California, was incorporated in Delaware in July 2010 and develops and operates proprietary technology applications to achieve this goal.

[Source: Passage 2]
Sources used: chunk_ids [455, 310, 477]
Latency: 725 ms

--------------------------------------------------------------------------------



## Part 3 Complete!

### What you built
| Item | Description |
|------|-------------|
| Weaviate collection `UberReportChunks` | Vector database storing all chunks + their embeddings |
| `dense_search()` | Semantic search via Weaviate vector similarity |
| `bm25_search()` | Keyword search via local BM25 index |
| `reciprocal_rank_fusion()` | Combines both rankings into one fused list |
| `cross_encoder_rerank()` | Final precision pass using a cross-encoder model |
| `query_librarian(question)` | The complete end-to-end pipeline — your required deliverable |

### Quick recap of the pipeline
1. Question comes in
2. Search Weaviate for semantically similar chunks (dense search)
3. Search local BM25 index for keyword-matching chunks
4. Merge both ranked lists with Reciprocal Rank Fusion
5. Re-score the top candidates with a cross-encoder for higher precision
6. Feed the best 3 chunks + question to GPT-4o-mini to generate a grounded answer
7. Return the answer along with which chunks were used (citations)

### Things to note for your Engineering Report
- The Librarian should rarely hallucinate numbers since it's reading directly from the
  source text, unlike The Intern which relies on memorized weights. This contrast is exactly
  what your "Hallucination Audit" section should highlight.
- Keep the test answers and latency numbers from Step 12 — you'll need them for the
  Part 4 Showdown comparison table.
- Note the latency: hybrid search + reranking + generation typically takes longer per query
  than a pure fine-tuned model's forward pass, which matters for your cost/speed tradeoff
  discussion.

### Next step → Part 4 (04_evaluation_arena.ipynb)
Run your golden_test_set.jsonl through both query_intern() and query_librarian(), score them
with ROUGE-L and LLM-as-a-Judge, measure latency, and build the final comparison table.

---
*Course: AI Engineer Essentials | Zuu Crew | Mini Project 01*
